In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [2]:
df = pd.read_parquet('../data/raw/sih_PR_2023_01.parquet')
print (f"Dados carregados: {len(df)} linhas")

Dados carregados: 1022423 linhas


In [3]:
df.describe

<bound method NDFrame.describe of         SP_GESTOR SP_UF SP_AA SP_MM  SP_CNES        SP_NAIH  SP_PROCREA  \
0          410000    41  2023    01  2729385  4123106805576  0301060088   
1          410000    41  2023    01  2729385  4123106805576  0301060088   
2          410000    41  2023    01  2729385  4123106805598  0303070129   
3          410000    41  2023    01  2729385  4123106805598  0303070129   
4          410000    41  2023    01  2729385  4123106805598  0303070129   
...           ...   ...   ...   ...      ...            ...         ...   
1022418    410000    41  2023    01  2742039  4123106972919  0301060088   
1022419    410000    41  2023    01  2742039  4123106972920  0301060088   
1022420    410000    41  2023    01  2742039  4123106972941  0301060088   
1022421    410000    41  2023    01  2742039  4123106972952  0301060088   
1022422    410000    41  2023    01  2742039  4123106972963  0301060088   

        SP_DTINTER SP_DTSAIDA SP_NUM_PR  ...        SP_PF_DOC    

In [4]:
print(f"Quantidade de colunas:{df.shape[1]}")
print(f"Registros: {len(df):,}")
print(f"Colunas:{df.columns.tolist()}")
print(f"Tipos: {df.dtypes}")


Quantidade de colunas:36
Registros: 1,022,423
Colunas:['SP_GESTOR', 'SP_UF', 'SP_AA', 'SP_MM', 'SP_CNES', 'SP_NAIH', 'SP_PROCREA', 'SP_DTINTER', 'SP_DTSAIDA', 'SP_NUM_PR', 'SP_TIPO', 'SP_CPFCGC', 'SP_ATOPROF', 'SP_TP_ATO', 'SP_QTD_ATO', 'SP_PTSP', 'SP_NF', 'SP_VALATO', 'SP_M_HOSP', 'SP_M_PAC', 'SP_DES_HOS', 'SP_DES_PAC', 'SP_COMPLEX', 'SP_FINANC', 'SP_CO_FAEC', 'SP_PF_CBO', 'SP_PF_DOC', 'SP_PJ_DOC', 'IN_TP_VAL', 'SEQUENCIA', 'REMESSA', 'SERV_CLA', 'SP_CIDPRI', 'SP_CIDSEC', 'SP_QT_PROC', 'SP_U_AIH']
Tipos: SP_GESTOR     object
SP_UF         object
SP_AA         object
SP_MM         object
SP_CNES       object
SP_NAIH       object
SP_PROCREA    object
SP_DTINTER    object
SP_DTSAIDA    object
SP_NUM_PR     object
SP_TIPO       object
SP_CPFCGC     object
SP_ATOPROF    object
SP_TP_ATO     object
SP_QTD_ATO    object
SP_PTSP       object
SP_NF         object
SP_VALATO     object
SP_M_HOSP     object
SP_M_PAC      object
SP_DES_HOS    object
SP_DES_PAC    object
SP_COMPLEX    object
SP_FIN

NAO USAREMOS TODAS AS 36 COLUNAS, SÃO MUITAS. PRECISA DE UM TRATAMENTO DE TIPO POIS TODAS ESTÃO COMO OBJECT, ATRAPALHANDO NOSSAS FUTURAS MÉTRICAS.

In [5]:
nulos = df.isnull().sum()
nulos_percent = (nulos / len(df))*100

df_nulos = pd.DataFrame({
    'Nulos' : nulos,
    'Percentual' : nulos_percent

}).sort_values('Nulos', ascending=False)
print(f"Valores Nulos por coluna:")
df_nulos

Valores Nulos por coluna:


,Nulos,Percentual
SP_GESTOR,0,0.0
SP_UF,0,0.0
SP_DES_HOS,0,0.0
SP_DES_PAC,0,0.0
SP_COMPLEX,0,0.0
SP_FINANC,0,0.0
SP_CO_FAEC,0,0.0
SP_PF_CBO,0,0.0
SP_PF_DOC,0,0.0
SP_PJ_DOC,0,0.0


NAO TEMOS COLUNAS COM VALORES NULOS

In [6]:
colunas_essenciais = [
       'SP_GESTOR',   # Código do Município
    'SP_CNES',     # ID do Hospital
    'SP_CIDPRI',   # Código CID-10 da doença
    'SP_DTINTER',  # Data de Internação
    'SP_DTSAIDA',  # Data de Saída
    'SP_VALATO',   # Valor do Ato Profissional
    'SP_M_PAC',    # Município do Paciente
    'SP_AA',       # Ano
    'SP_MM'        # Mês
]

df_filtrado = df[colunas_essenciais].copy()
df_filtrado

,SP_GESTOR,SP_CNES,SP_CIDPRI,SP_DTINTER,SP_DTSAIDA,SP_VALATO,SP_M_PAC,SP_AA,SP_MM
0,410000,2729385,I64,20230126,20230127,0.00,411270,2023,01
1,410000,2729385,I64,20230126,20230127,0.00,411270,2023,01
2,410000,2729385,K838,20230128,20230130,41.95,410980,2023,01
3,410000,2729385,K838,20230128,20230130,16.00,410980,2023,01
4,410000,2729385,K838,20230128,20230130,16.78,410980,2023,01
...,...,...,...,...,...,...,...,...,...
1022418,410000,2742039,T012,20230113,20230114,33.34,411930,2023,01
1022419,410000,2742039,I64,20230105,20230105,33.34,411930,2023,01
1022420,410000,2742039,T012,20230110,20230110,33.34,411930,2023,01
1022421,410000,2742039,T012,20230103,20230104,33.34,411930,2023,01


In [8]:
df_filtrado['SP_DTINTER'] = pd.to_datetime(df_filtrado['SP_DTINTER'], format='%Y%m%d', errors='coerce')
df_filtrado['SP_DTSAIDA'] = pd.to_datetime(df_filtrado['SP_DTSAIDA'], format='%Y%m%d', errors ='coerce')

df_filtrado['SP_GESTOR'] = pd.to_numeric(df_filtrado['SP_GESTOR'], errors = 'coerce' )

df_filtrado['SP_AA'] = pd.to_numeric(df_filtrado['SP_AA'], errors='coerce').fillna(0).astype(int)
df_filtrado['SP_MM'] = pd.to_numeric(df_filtrado['SP_MM'], errors='coerce').fillna(0).astype(int)

df_filtrado.dtypes

,0
SP_GESTOR,int64
SP_CNES,object
SP_CIDPRI,object
SP_DTINTER,datetime64[ns]
SP_DTSAIDA,datetime64[ns]
SP_VALATO,object
SP_M_PAC,object
SP_AA,int64
SP_MM,int64
